# NNs: Generalization, Regularization, and Modeling

In the previous lectures, we introduced feed-forward neural networks as flexible models trained by empirical risk minimization. We defined a network as a composition of layers, discussed output layers for regression and classification, and described how backpropagation computes gradients for mini-batch SGD.

In this lecture, we focus on the practical modeling questions that arise when using neural networks:

- How large should the network be?
- How do we recognize overfitting?
- How do we regularize a neural network?
- How do we choose training settings?
- How do we evaluate the final model?

## Model Complexity in Neural Networks

The flexibility of a neural network depends on several modeling choices.

Important choices include:

- the number of hidden layers
- the number of units in each hidden layer
- the activation functions
- the output layer
- the total number of parameters
- the amount of training
- the amount of regularization

A neural network with more layers or more units can usually represent more complicated functions. This can be useful when the true relationship between $x$ and $y$ is complex. But it also creates more opportunities to overfit the training data.

### Depth and Width

The **depth** of a network refers to the number of layers in the composition. If

$$
s_\theta(x)=
s_M(s_{M-1}(\cdots s_2(s_1(x))\cdots)),
$$

then the network has $M$ layers in this composition. Sometimes people count only the hidden layers.

The **width** of a layer refers to the number of units in that layer. If layer $i$ maps

$$
s_i: \mathbb{R}^{d_{i-1}} \to \mathbb{R}^{d_i},
$$

then layer $i$ has width $d_i$.

For a dense layer,

$$
h^{(i)}=
\phi_i(W^{(i)}h^{(i-1)} + b^{(i)}),
$$

where

$$
W^{(i)} \in \mathbb{R}^{d_i \times d_{i-1}}
$$

and

$$
b^{(i)} \in \mathbb{R}^{d_i}.
$$

So layer $i$ has

$$
d_i d_{i-1} + d_i
$$

parameters.

This is one reason neural networks can become large quickly. Even a moderate number of hidden units can produce many weights.

### Parameter Count Example

Suppose a network has:

- $D=20$ input features
- one hidden layer with $H=50$ hidden units
- one scalar output

The hidden layer has

$$
50 \cdot 20 + 50 = 1050
$$

parameters.

The output layer has

$$
1 \cdot 50 + 1 = 51
$$

parameters.

So the full network has

$$
1050 + 51 = 1101
$$

parameters.

This is already much larger than a simple linear regression with $20$ predictors. More width and more depth can increase the parameter count very quickly.

## Regularization

One way to counteract potential over-fitting is via **regularization**. 

In neural networks, regularization can take several forms:

- explicit penalties on weights ("weight decay")
- stopping training early ("early stopping")
- randomly dropping hidden units during training ("dropout")
- choosing smaller architectures

### Weight Decay

Weight decay is the neural network analogue of ridge-style regularization.

Instead of minimizing only the empirical risk,

$$
\hat{R}(\theta),
$$

we add a penalty on the size of the weights:

$$
\hat{R}(\theta) + \lambda \Omega(\theta).
$$

A common choice is an $L_2$ penalty on the weights $\Omega(\theta) = ||\theta||_2^2$.

The main ideas are:

- Weight decay discourages very large weights.
- It can make the fitted function less extreme.
- It can improve validation performance.
- It connects directly to ridge regression and elastic net ideas from earlier lectures.

In software, weight decay is often implemented either as an optimizer option or as a per-layer-level regularizer. Both potentially have their places. 

### Early Stopping

Another approach is **early stopping**. This is one of the simplest and most useful regularization methods.

The idea is:

1. Train the model over many epochs.
2. Track validation loss after each epoch.
3. Stop training when validation loss stops improving.
4. Keep the parameter values from the best validation epoch.

Early stopping is useful because overfitting often appears as continued improvement in training loss but worsening validation loss.

### Dropout

Dropout is another common neural network regularization method.

During training, dropout randomly sets some hidden-unit **outputs** to zero. This means the network cannot rely too heavily on any one hidden unit. It is typically turned off, or replaced by a deterministic scaling convention, at prediction time.

For example, if we apply dropout with probability $0.2$, then during each training step, each hidden-unit output has a $20\%$ chance of being set to zero. The particular units dropped out change from one training step to the next, so the network is trained under many slightly different versions of itself.

This has a regularizing effect. It discourages hidden units from becoming too specialized or too dependent on one another. Instead, the network is pushed to learn representations that remain useful even when some units are temporarily removed. In some ways (will discuss later) its almost like averaging over a class of models with hidden units randomly set to zero. 

### Smaller Networks as Regularization

Another simple form of regularization is choosing a smaller model.

For example, instead of using three hidden layers with hundreds of units, we might start with one hidden layer and a modest number of units.

A practical strategy is:

1. Start with a small network.
2. Check training and validation performance.
3. Increase complexity only if the model appears to be underfitting.


## Practical Optimization Issues

Even after choosing an architecture and loss function, neural network training depends on several practical choices.

Important training choices include:

- learning rate
- number of epochs
- batch size
- initialization
- input scaling
- optimizer
- monitoring training and validation loss

### Learning Rate

The learning rate controls how large each parameter update is. For mini-batch SGD,

$$
\theta^{(t+1)}=
\theta^{(t)}-
\eta \nabla_\theta \hat{R}_{B_t}(\theta^{(t)}),
$$

where $\eta$ is the learning rate.

If $\eta$ is too small, training may be very slow.

If $\eta$ is too large, training may be unstable. The loss may jump around or diverge.

The learning rate is often one of the most important hyperparameters. More advaned optimizers will attempt to adaptively set this learning rate, or use some schedule of learning rates. 

### Number of Epochs

An epoch is one full pass through the training data. Training for too few epochs can lead to underfitting. Training for too many epochs can lead to overfitting. This is why we track validation loss and often use early stopping.

### Batch Size

The batch size is the number of training observations used in each gradient update. Small batch sizes give noisy but cheap gradient estimates. Large batch sizes give smoother but more expensive gradient estimates.

Typical values are often $32$, $64$, $128$, or $256$ (powers of $2$).

### Initialization

Neural networks usually initialize $\theta$ randomly. Initialization matters because neural network objectives are usually nonconvex. Different starting values can lead to different fitted networks. In practice, software usually provides reasonable default initialization schemes, although there has been some research showing that this can matter. 

### Scaling Inputs

Neural networks are sensitive to input scale.

If one feature has values near $0$ and another feature has values in the thousands, gradient-based optimization can become harder. A common preprocessing step is to standardize numerical predictors using the training data:

$$
x_j^{\text{scaled}}=
\frac{x_j - \bar{x}_j}{s_j}.
$$

The same transformation is then applied to the validation and test sets.

**Important point:** preprocessing parameters, such as means and standard deviations, should be learned from the training data only. Otherwise, information from the validation or test data can leak into the training process.

### Optimizer Variants

So far, we have focused on gradient descent and mini-batch SGD. Many neural networks use optimizer variants that modify the basic update rule.

Common examples include:

- SGD with momentum
- RMSProp
- Adam

We won't have time to go into these extensively, but they all offer potentially more advanced ways to update the learning rate and gradient. **Adam** is a common default choice in many applications, while **SGD with momentum** is also widely used. Its difficult to know, in advance, which will work better. 

# Example: MNIST Classification in Keras

In this example, we use a simple feed-forward network. This means we flatten each image into a vector and then apply "dense" layers. This is not the best possible architecture for images, because it ignores the spatial structure of the pixels. But it is a useful example for understanding standard feed-forward networks for multiclass classification.

We will use **Keras** because it gives a compact, high-level interface for defining and training neural networks, while still fitting naturally with the `fit` / `predict` workflow students have already seen in scikit-learn. Alternatives include **PyTorch**, which gives more direct control over model code and training loops, and **TensorFlow**, which is a broader machine-learning platform often used for both training and deployment. (Keras 3 can also run on top of TensorFlow, JAX, or PyTorch backends.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import keras
from keras import layers, regularizers

## Load the MNIST Data

Keras provides a convenient MNIST loader. The data are already split into training and test sets.

In [ ]:
(x_train_full0, y_train_full0), (x_test0, y_test0) = keras.datasets.mnist.load_data()

print(x_train_full0.shape)
print(y_train_full0.shape)
print(x_test0.shape)
print(y_test0.shape)

Each image is stored as a $28 \times 28$ array. The labels are integers from $0$ to $9$.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(8, 4))
axes = axes.ravel()

for i in range(10):
    axes[i].imshow(x_train_full0[i], cmap="gray")
    axes[i].set_title(f"label = {y_train_full0[i]}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## Preprocess the Data

Pixel values are integers between $0$ and $255$. We rescale them to be between $0$ and $1$.

We also create a validation set from the original training data.

In [ ]:
from sklearn.model_selection import train_test_split

x_train_full = x_train_full0.astype("float32") / 255.0
x_test = x_test0.astype("float32") / 255.0

# Choose how much data to use
n_total = None      # set to None to use all 60,000 training observations
n_test = None      # set to None to use all test observations
val_size = 0.20    # fraction of selected training data used for validation

# Optionally use only a subset of the training data
if n_total is None:
    n_total = 60000
    
x_train_full = x_train_full[:n_total]
y_train_full = y_train_full0[:n_total]

# Random, stratified train/validation split
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=val_size,
    random_state=654654,
    stratify=y_train_full
)

# Optionally subset test data
if n_test is None:
    n_test = 10000
    
x_test = x_test[:n_test]
y_test = y_test0[:n_test]

print("Training:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

We keep the labels as integers. This lets us use **sparse categorical cross-entropy**. (If we instead converted the labels to one-hot vectors, we would use categorical cross-entropy. They are functionally identical.)

## Define a Feed-Forward Model

The model has the following structure:

1. Flatten the $28 \times 28$ image into a vector of length $784$.
2. Apply a dense hidden layer with ReLU activation.
3. Apply dropout for regularization.
4. Apply another dense hidden layer with ReLU activation.
5. Apply a final dense layer with $10$ units and softmax activation.

The final softmax layer returns one predicted probability for each digit class.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4), #l2 regularization
    ),
    layers.Dropout(0.1), #dropout regularization
    layers.Dense(64, activation="relu"), 
    layers.Dense(10, activation="softmax"),
])

model.summary()

## Compile the Model

For multiclass classification, we use cross-entropy loss.

Because the labels are stored as integers, we use `sparse_categorical_crossentropy`.

We track accuracy as an additional metric.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001), #adam with specified lr
    loss="sparse_categorical_crossentropy", # actual loss
    metrics=["accuracy"], # what we're tracking
)

## Fit the Model

We use mini-batches, validation data, and early stopping.

Early stopping monitors validation loss and restores the weights from the epoch with the best validation loss.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10, # how long to wait without improvement
    restore_best_weights=True,
)

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=100,
    batch_size=128,
    callbacks=[early_stopping],
)

## Plot Training and Validation Curves

The `history` object stores the training and validation loss and accuracy values from each epoch.

In [ ]:
history_dict = history.history

epochs = range(1, len(history_dict["loss"]) + 1)

plt.plot(epochs, history_dict["loss"], label="Training loss")
plt.plot(epochs, history_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.plot(epochs, history_dict["accuracy"], label="Training accuracy")
plt.plot(epochs, history_dict["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Evaluate on the Test Set

The test set should be used after model selection is complete. It gives a final estimate of generalization performance.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## Inspect Predictions

The model outputs a probability vector with $10$ entries. The predicted class is the digit with the largest predicted probability.

In [ ]:
y_prob = model.predict(x_test[:10])
y_pred = np.argmax(y_prob, axis=1)

print("Predicted labels:", y_pred)
print("True labels:     ", y_test[:10])

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(8, 4))
axes = axes.ravel()

for i in range(10):
    axes[i].imshow(x_test[i], cmap="gray")
    axes[i].set_title(f"pred={y_pred[i]}, true={y_test[i]}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

We can also look at a confusion matrix:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_prob_test = model.predict(x_test, verbose=0)
y_pred_test = np.argmax(y_prob_test, axis=1)

cm = confusion_matrix(y_test, y_pred_test)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.show()

This model has 109,386 trainable parameters, with 32,000 training observations and 8,000 validation observations. That is a lot of parameters for a model that is still fairly simple: just flattened images, dense layers, ReLU activations, dropout, and weight decay.

Neural networks are often **over-parameterized**, meaning they may have a very large number of parameters relative to the apparent complexity of the task, and sometimes even relative to the number of training examples. 

# Over-Parameterized Models and Generalization

The MNIST example may feel surprising at first. The neural network has a large number of trainable parameters, but it still performs well on the validation and test data. This is different from the simplest version of the bias-variance story, where we might expect a highly flexible model to overfit badly.

The key point is that **over-parameterized does not automatically mean overfit**. A model is over-parameterized when it has many parameters, often enough to fit the training data extremely well. But overfitting is about generalization: whether the model performs poorly on new data. Those are related ideas, but they are not the same.

There are several reasons why an over-parameterized neural network can still generalize:

- The model architecture imposes some structure, even if it has many parameters.
- Regularization methods such as weight decay and dropout can discourage overly complex fitted functions.
- Optimization can have an implicit regularizing effect. Gradient-based methods do not necessarily find an arbitrary solution that fits the training data; they tend to prefer certain kinds of solutions.

This last point is important. In over-parameterized problems, there may be many different parameter settings that fit the training data. The training algorithm helps determine which one we actually get.

## Double Descent

This connects to a modern phenomenon called **double descent**.

In the classical bias-variance picture, test error often behaves like a U-shaped curve as model complexity increases. A very simple model underfits. A moderately flexible model performs better. A very flexible model overfits.

Double descent says that this is not always the full story. In some modern machine-learning settings, test error may:

1. decrease at first as model complexity increases,
2. increase near the point where the model is just flexible enough to interpolate the training data,
3. then decrease again as the model becomes even more over-parameterized.

The point where the model first becomes able to fit the training data nearly perfectly is often called the **interpolation threshold**. Around this threshold, generalization can be poor. But beyond this point, larger models may again generalize well.

This does not mean larger models are always better. It means parameter count alone is not enough to determine whether a model will overfit, and parameter count isn't always the best measure of model complexity in practice. 

### A Simple Example: Minimum-Norm Polynomial Regression

To build intuition for this, we will look at a simpler setting: polynomial regression.

Suppose we fit a polynomial model with more features than observations. In that case, there may be infinitely many polynomials that interpolate the training data exactly. *If many models fit the training data perfectly, which one does the training procedure choose?*

In over-parameterized linear regression, gradient descent from a small or zero initialization has a special implicit bias: it selects the **minimum-norm interpolating solution** i.e. **ridgeless regression**. The optimization procedure itself favors a particular solution among the many possible interpolating solutions.

Consider this below:

In [ ]:
from numpy.polynomial.legendre import legvander

rng = np.random.default_rng(565)

def f(x):
    return np.sin(2 * np.pi * x) + 0.5 * x

def min_norm_poly_fit(x_train, y_train, degree):
    Phi = legvander(x_train, degree)  # shape: (n, degree + 1)

    # Minimum-norm least squares solution
    # Works in both underparameterized and overparameterized regimes.
    beta = np.linalg.pinv(Phi) @ y_train

    return beta

def predict(x, beta):
    degree = len(beta) - 1
    Phi = legvander(x, degree)
    return Phi @ beta

In [ ]:
n_train = 40
n_test = 5000
noise_sd = 0.1

degrees = np.arange(1, 200)
n_trials = 50

test_errors = []

x_test = rng.uniform(-1, 1, n_test)
y_test_true = f(x_test)

for degree in degrees:
    errs = []

    for _ in range(n_trials):
        x_train = rng.uniform(-1, 1, n_train)
        y_train = f(x_train) + rng.normal(0, noise_sd, n_train)

        beta = min_norm_poly_fit(x_train, y_train, degree)
        y_pred = predict(x_test, beta)

        mse = np.mean((y_pred - y_test_true) ** 2)
        errs.append(mse)

    test_errors.append(np.median(errs))

test_errors = np.array(test_errors)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(degrees + 1, test_errors, linewidth=2)

plt.axvline(
    n_train,
    linestyle="--",
    label="Interpolation threshold: parameters = training samples"
)

plt.yscale("log")
plt.xlabel("Number of polynomial features")
plt.ylabel("Median test MSE")
plt.title("Double Descent in Minimum-Norm Polynomial Regression")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
n_train = 40
noise_sd = 0.25

x_train = rng.uniform(-1, 1, n_train)
y_train = f(x_train) + rng.normal(0, noise_sd, n_train)

x_grid = np.linspace(-1, 1, 1000)
y_true = f(x_grid)

# Number of parameters = degree + 1.
# Interpolation threshold is roughly degree + 1 = n_train.
degrees_to_plot = [3, 10, n_train - 1, 199]

# ----------------------------
# Plot interpolators
# ----------------------------
plt.figure(figsize=(10, 6))

plt.plot(
    x_grid,
    y_true,
    linewidth=3,
    label="true function"
)

plt.scatter(
    x_train,
    y_train,
    s=35,
    zorder=5,
    label="noisy training data"
)

for degree in degrees_to_plot:
    beta = min_norm_poly_fit(x_train, y_train, degree)
    y_hat = predict(x_grid, beta)

    plt.plot(
        x_grid,
        y_hat,
        linewidth=1.8,
        label=f"degree {degree} ({degree + 1} params)"
    )

plt.axhline(0, linewidth=0.5)
plt.ylim(-4, 4)  # Keeps extreme oscillations from dominating the plot.
plt.xlabel("x")
plt.ylabel("y")
plt.title("Minimum-Norm Polynomial Fits at Different Degrees")
plt.legend()
plt.tight_layout()
plt.show()